In [12]:
import importlib
import sys
import torch
import pickle
import os
from tqdm.notebook import tqdm

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')
sys.path.insert(0, '../../../..')
sys.path.insert(0, '../../../../..')

from model.dropout_uncertainty_enc_dec_LSTM.dropout_uncertainty_model import DropoutUncertaintyEncoderDecoderLSTM


In [13]:
# Load model
file_path_model = '../../../training_variational_dropout/BPIC17/BPIC_2017_setting_2.pkl'
model = DropoutUncertaintyEncoderDecoderLSTM.load(file_path_model, dropout=0.1)

# Load the dataset
file_path_data_set = '../../../../../encoded_data/BPIC17/BPIC_2017_all_5_val.pkl'
bpic_17_test_dataset = torch.load(file_path_data_set, weights_only=False)

print(f"Model loaded")
print(f"Dataset loaded: {len(bpic_17_test_dataset)} cases")


Data set categories:  ([('concept:name', 28, {'A_Accepted': 1, 'A_Cancelled': 2, 'A_Complete': 3, 'A_Concept': 4, 'A_Create Application': 5, 'A_Denied': 6, 'A_Incomplete': 7, 'A_Pending': 8, 'A_Submitted': 9, 'A_Validating': 10, 'EOS': 11, 'O_Accepted': 12, 'O_Cancelled': 13, 'O_Create Offer': 14, 'O_Created': 15, 'O_Refused': 16, 'O_Returned': 17, 'O_Sent (mail and online)': 18, 'O_Sent (online only)': 19, 'W_Assess potential fraud': 20, 'W_Call after offers': 21, 'W_Call incomplete files': 22, 'W_Complete application': 23, 'W_Handle leads': 24, 'W_Personal Loan collection': 25, 'W_Shortened completion ': 26, 'W_Validate application': 27}), ('Action', 7, {'Created': 1, 'Deleted': 2, 'EOS': 3, 'Obtained': 4, 'Released': 5, 'statechange': 6}), ('org:resource', 150, {'EOS': 1, 'User_1': 2, 'User_10': 3, 'User_100': 4, 'User_101': 5, 'User_102': 6, 'User_103': 7, 'User_104': 8, 'User_105': 9, 'User_106': 10, 'User_107': 11, 'User_108': 12, 'User_109': 13, 'User_11': 14, 'User_110': 15, 'U

/Users/leonurny/.local/share/virtualenvs/Probabilistic_Suffix_Prediction_U-ED-LSTM_-32bEAP25/lib/python3.13/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator OrdinalEncoder from version 1.5.2 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/leonurny/.local/share/virtualenvs/Probabilistic_Suffix_Prediction_U-ED-LSTM_-32bEAP25/lib/python3.13/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.2 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/leonurny/.local/share/virtualenvs/Probabilis

In [14]:
attack_dataset = '../../../../../encoded_data/BPIC17/val.pkl'
predefined_dataset = torch.load(attack_dataset, weights_only=False)

In [15]:
# Sample 10% of observations for faster inference
import random

# Set random seed for reproducibility (optional)
random.seed(17)

# Get all keys from both datasets
all_keys_orig = list(predefined_dataset.keys())

# Calculate 5% sample size
sample_size = max(1, int(len(all_keys_orig) * 0.05))

# Randomly sample 10% of the keys
sampled_keys = random.sample(all_keys_orig, sample_size)

# Create new dictionaries with only sampled keys
predefined_dataset = {key: predefined_dataset[key] for key in sampled_keys}

print(f"Sampled {len(sampled_keys)} observations ({len(sampled_keys)/len(all_keys_orig)*100:.1f}%) from {len(all_keys_orig)} total observations")
print(f"Original dataset now has {len(predefined_dataset)} entries")

Sampled 8580 observations (5.0%) from 171611 total observations
Original dataset now has 8580 entries


In [16]:
# Import and reload the adversarial attack module
import evaluation.adversarial_attack
importlib.reload(evaluation.adversarial_attack)
from evaluation.adversarial_attack import GradientAscentAttacker

# Create the gradient ascent attacker
attacker = GradientAscentAttacker(
    model=model,
    dataset=bpic_17_test_dataset,
    concept_name='concept:name',
    growing_num_values=['case_elapsed_time'],
    all_cat=['concept:name', 'org:resource', 'lifecycle:transition'],
    all_num=['case_elapsed_time', 'event_elapsed_time'],
    dataset_predefined_prefixes=predefined_dataset
)

print("GradientAscentAttacker initialized")


GradientAscentAttacker initialized


In [17]:
# Configure attack parameters
max_iterations = 15  # Maximum gradient ascent steps per attack
embedding_step_size = 0.3    # Learning rate for embedding perturbations
time_step_size = 0.001        # Learning rate for time feature perturbations
embedding_epsilon = 10.0     # Maximum allowed perturbation for embeddings (L_inf norm)
time_epsilon = 0.1           # Maximum allowed perturbation for time features (L_inf norm)
early_stop = True            # Stop when prediction becomes wrong

print(f"Attack parameters:")
print(f"  Max iterations: {max_iterations}")
print(f"  Embedding step size: {embedding_step_size}")
print(f"  Time step size: {time_step_size}")
print(f"  Embedding epsilon: {embedding_epsilon}")
print(f"  Time epsilon: {time_epsilon}")
print(f"  Early stop: {early_stop}")


Attack parameters:
  Max iterations: 15
  Embedding step size: 0.3
  Time step size: 0.001
  Embedding epsilon: 10.0
  Time epsilon: 0.1
  Early stop: True


In [18]:
# Function to save results in chunks
def save_chunk(results, chunk_number):
    filename = os.path.join(output_dir, f'gradient_ascent_attack_part_{chunk_number:04d}.pkl')
    with open(filename, 'wb') as f:
        pickle.dump(results, f)
    print(f"Saved {len(results)} results to {filename}")

# Set output directory
output_dir = '../../../../../evaluation_results/robustness/BPIC17/gradient_ascent_attack/'
os.makedirs(output_dir, exist_ok=True)

save_every = 50  # Save every N successful attacks
print(f"Output directory: {output_dir}")
print(f"Saving every {save_every} attacks")


Output directory: ../../../../../evaluation_results/robustness/BPIC17/gradient_ascent_attack/
Saving every 50 attacks


In [19]:
# Perform gradient ascent attacks on all predefined prefixes
print("Starting gradient ascent attacks...")
print(f"Total prefix-suffix pairs to attack: {len(predefined_dataset)}")

results = attacker.attack_predefined_prefixes(
    max_iterations=max_iterations,
    embedding_step_size=embedding_step_size,
    time_step_size=time_step_size,
    embedding_epsilon=embedding_epsilon,
    time_epsilon=time_epsilon,
    early_stop=early_stop,
    attackable_features="all",
    enable_time_shifting=True
)

print(f"\nAttack completed!")
print(f"Total attacks performed: {len(results)}")
print(f"Successful attacks: {sum(1 for r in results.values() if r['success'])}")
print(f"Failed attacks: {sum(1 for r in results.values() if not r['success'])}")


Starting gradient ascent attacks...
Total prefix-suffix pairs to attack: 8580


Performing gradient ascent attacks: 100%|██████████| 8580/8580 [10:34<00:00, 13.53it/s] 


Attack completed!
Total attacks performed: 92
Successful attacks: 0
Failed attacks: 92


In [20]:
# Save results
if len(results) > 0:
    if len(results) <= save_every:
        filename = os.path.join(output_dir, 'gradient_ascent_attack_all.pkl')
        with open(filename, 'wb') as f:
            pickle.dump(results, f)
        print(f"Saved all {len(results)} results to {filename}")
    else:
        results_list = list(results.items())
        for i in range(0, len(results_list), save_every):
            chunk = dict(results_list[i:i+save_every])
            chunk_number = (i // save_every) + 1
            save_chunk(chunk, chunk_number)
        print(f"Saved {len(results)} results in chunks")
else:
    print("No results to save")


Saved 50 results to ../../../../../evaluation_results/robustness/BPIC17/gradient_ascent_attack/gradient_ascent_attack_part_0001.pkl
Saved 42 results to ../../../../../evaluation_results/robustness/BPIC17/gradient_ascent_attack/gradient_ascent_attack_part_0002.pkl
Saved 92 results in chunks


In [21]:
# Print summary statistics
if len(results) > 0:
    successful_attacks = [r for r in results.values() if r['success']]
    failed_attacks = [r for r in results.values() if not r['success']]
    
    print("\n=== Attack Summary ===")
    print(f"Total attacks: {len(results)}")
    print(f"Successful attacks: {len(successful_attacks)}")
    print(f"Failed attacks: {len(failed_attacks)}")
    
    if successful_attacks:
        num_steps = [r['num_steps'] for r in successful_attacks]
        print(f"\nSuccessful attack statistics:")
        print(f"  Average steps: {sum(num_steps) / len(num_steps):.2f}")
        print(f"  Min steps: {min(num_steps)}")
        print(f"  Max steps: {max(num_steps)}")
    
    if failed_attacks:
        num_steps_failed = [r['num_steps'] for r in failed_attacks]
        print(f"\nFailed attack statistics:")
        print(f"  Average steps: {sum(num_steps_failed) / len(num_steps_failed):.2f}")
        print(f"  All reached max iterations: {all(n == max_iterations for n in num_steps_failed)}")
else:
    print("No results to summarize")



=== Attack Summary ===
Total attacks: 92
Successful attacks: 0
Failed attacks: 92

Failed attack statistics:
  Average steps: 15.00
  All reached max iterations: True


In [22]:
# Example: Inspect a few attack results
if len(results) > 0:
    print("\n=== Example Attack Results ===")
    
    successful = [(k, v) for k, v in results.items() if v['success']]
    if successful:
        print(f"\nFirst successful attack:")
        (case_id, prefix_len), result = successful[0]
        print(f"  Case ID: {case_id}, Prefix Length: {prefix_len}")
        print(f"  Steps taken: {result['num_steps']}")
        print(f"  Original suffix length: {len(result['original_suffix'])}")
        print(f"  Perturbed suffix length: {len(result['perturbed_suffix'])}")
        
        if result['original_suffix'] and result['perturbed_suffix']:
            orig_activities = [e.get('concept:name', 'N/A') for e in result['original_suffix']]
            pert_activities = [e.get('concept:name', 'N/A') for e in result['perturbed_suffix']]
            print(f"  Original activities: {orig_activities}")
            print(f"  Perturbed activities: {pert_activities}")
    
    failed = [(k, v) for k, v in results.items() if not v['success']]
    if failed:
        print(f"\nFirst failed attack:")
        (case_id, prefix_len), result = failed[0]
        print(f"  Case ID: {case_id}, Prefix Length: {prefix_len}")
        print(f"  Steps taken: {result['num_steps']}")
        print(f"  Note: Attack did not succeed within {max_iterations} iterations")
else:
    print("No results to inspect")



=== Example Attack Results ===

First failed attack:
  Case ID: Application_1139122833, Prefix Length: 28
  Steps taken: 15
  Note: Attack did not succeed within 15 iterations


In [23]:
# Print before/after prefix and suffix for all attack candidates
if len(results) > 0:
    print("\n" + "="*80)
    print("BEFORE/AFTER PREFIX AND SUFFIX FOR ALL ATTACK CANDIDATES")
    print("="*80)
    
    for idx, ((case_id, prefix_len), result) in enumerate(results.items(), 1):
        print(f"\n{'='*80}")
        print(f"Attack #{idx}: Case ID: {case_id}, Prefix Length: {prefix_len}")
        print(f"Status: {'SUCCESS' if result['success'] else 'FAILED'}")
        print(f"Steps taken: {result['num_steps']}")
        print(f"{'='*80}")
        
        original_prefix_readable = attacker.case_to_readable(
            (result['original_prefix'][0], result['original_prefix'][1]), 
            prune_eos=True
        )
        
        perturbed_prefix_readable = attacker.case_to_readable(
            (result['perturbed_prefix'][0], result['perturbed_prefix'][1]), 
            prune_eos=True
        )
        
        print(f"\n--- PREFIX COMPARISON (Length: {len(original_prefix_readable)}) ---")
        max_prefix_len = max(len(original_prefix_readable), len(perturbed_prefix_readable))
        for i in range(max_prefix_len):
            print(f"\n  Event {i+1}:")
            orig_event = original_prefix_readable[i] if i < len(original_prefix_readable) else {}
            pert_event = perturbed_prefix_readable[i] if i < len(perturbed_prefix_readable) else {}
            
            all_keys = set(orig_event.keys()) | set(pert_event.keys())
            
            for key in sorted(all_keys):
                orig_value = orig_event.get(key, 'N/A')
                pert_value = pert_event.get(key, 'N/A')
                
                if orig_value != pert_value:
                    print(f"    {key} = [{orig_value}] -> [{pert_value}] \u26a0\ufe0f CHANGED")
                else:
                    print(f"    {key} = [{orig_value}], [{pert_value}]")
        
        print(f"\n--- SUFFIX COMPARISON ---")
        orig_suffix = result['original_suffix']
        pert_suffix = result['perturbed_suffix']
        max_suffix_len = max(len(orig_suffix), len(pert_suffix))
        
        for i in range(max_suffix_len):
            orig_event = orig_suffix[i] if i < len(orig_suffix) else {}
            pert_event = pert_suffix[i] if i < len(pert_suffix) else {}
            
            all_keys = set(orig_event.keys()) | set(pert_event.keys())
            
            print(f"\n  Event {i+1}:")
            for key in sorted(all_keys):
                orig_value = orig_event.get(key, 'N/A')
                pert_value = pert_event.get(key, 'N/A')
                
                if orig_value != pert_value:
                    print(f"    {key} = [{orig_value}] -> [{pert_value}] \u26a0\ufe0f CHANGED")
                else:
                    print(f"    {key} = [{orig_value}], [{pert_value}]")
        
        orig_prefix_activities = [e.get('concept:name', 'N/A') for e in original_prefix_readable]
        pert_prefix_activities = [e.get('concept:name', 'N/A') for e in perturbed_prefix_readable]
        orig_suffix_activities = [e.get('concept:name', 'N/A') for e in result['original_suffix']]
        pert_suffix_activities = [e.get('concept:name', 'N/A') for e in result['perturbed_suffix']]
        
        print(f"\n--- ACTIVITY SEQUENCE SUMMARY ---")
        print(f"Prefix activities: {orig_prefix_activities} -> {pert_prefix_activities}")
        print(f"Suffix activities: {orig_suffix_activities} -> {pert_suffix_activities}")
        
        prefix_changed = orig_prefix_activities != pert_prefix_activities
        print(f"\nPrefix changed: {prefix_changed}")
        if prefix_changed:
            print("  Positions changed:")
            for i, (orig, pert) in enumerate(zip(orig_prefix_activities, pert_prefix_activities)):
                if orig != pert:
                    print(f"    Position {i+1}: '{orig}' -> '{pert}'")
        
        suffix_changed = orig_suffix_activities != pert_suffix_activities
        print(f"Suffix changed: {suffix_changed}")
        if suffix_changed:
            print("  Positions changed:")
            min_len = min(len(orig_suffix_activities), len(pert_suffix_activities))
            for i in range(min_len):
                if orig_suffix_activities[i] != pert_suffix_activities[i]:
                    print(f"    Position {i+1}: '{orig_suffix_activities[i]}' -> '{pert_suffix_activities[i]}'")
            if len(orig_suffix_activities) != len(pert_suffix_activities):
                print(f"  Length difference: {len(orig_suffix_activities)} vs {len(pert_suffix_activities)}")
        
        print(f"\n{'-'*80}")
    
    print(f"\n{'='*80}")
    print(f"Total attacks printed: {len(results)}")
    print(f"{'='*80}")
else:
    print("No results to print")



BEFORE/AFTER PREFIX AND SUFFIX FOR ALL ATTACK CANDIDATES

Attack #1: Case ID: Application_1139122833, Prefix Length: 28
Status: FAILED
Steps taken: 15

--- PREFIX COMPARISON (Length: 28) ---

  Event 1:
    Accepted = [None], [None]
    Action = [Created], [Created]
    CreditScore = [318.75640869140625], [318.75640869140625]
    EventOrigin = [Application], [Application]
    FirstWithdrawalAmount = [8474.0869140625], [8474.0869140625]
    MonthlyCost = [282.88421534161523], [282.88421534161523]
    NumberOfTerms = [82.9709701538086], [82.9709701538086]
    Selected = [None], [None]
    case:ApplicationType = [New credit], [New credit]
    case:LoanGoal = [Car], [Car]
    case:RequestedAmount = [7500.000388928573], [7500.000388928573]
    case_elapsed_time = [0.031988635775633156] -> [-0.024187054485082626] ⚠️ CHANGED
    concept:name = [A_Create Application], [A_Create Application]
    day_in_week = [4.000000103613218] -> [3.976823093695895] ⚠️ CHANGED
    event_elapsed_time = [51002